In [ ]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + STT 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


# ── STT 세그먼트 복원 함수 ──
def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


# ── 단일 영상 로드 함수 ──
def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        return None

    duration = data.get("duration", max(inst["end_sec"] for inst in instances))

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })

    # STT 로드
    video_name = os.path.basename(path)[:-5]  # .json 제거
    stt_path = os.path.join(STT_DIR, channel, video_name + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass

    return {
        "channel": channel,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


# ── 병렬 로드 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        if result is not None:
            channel_set.add(result["channel"])
            samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── STT 통계 ──
stt_counts = [len(s["stt_segments"]) for s in samples]
stt_counts = np.array(stt_counts)
print(f"\n📊 영상별 STT 세그먼트 수")
print(f"  mean: {stt_counts.mean():.1f}  median: {np.median(stt_counts):.0f}  "
      f"p99: {np.percentile(stt_counts, 99):.0f}  max: {stt_counts.max()}")

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

inst_counts = np.array([len(s["instances"]) for s in samples])
print(f"\n✅ 영상: {len(samples):,}개  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

In [ ]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 8
MASK_RATIO = 1.0


class BLTDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb, mask_ratio=MASK_RATIO):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb
        self.mask_ratio = mask_ratio

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        n = len(s["instances"])
        duration = max(s["duration"], 0.1)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)

        # ── 텔롭 인스턴스 ──
        text_embs = []
        cond_feats = []
        gt_x, gt_y, gt_w, gt_h = [], [], [], []
        inst_starts, inst_ends = [], []

        for inst in s["instances"]:
            text_embs.append(self.text2emb.get(inst["text"], ZERO_EMB))
            cond_feats.append([
                inst["text_len"] / 200.0,
                inst["start"] / duration,
                inst["end"] / duration,
                (inst["end"] - inst["start"]) / duration,
            ])
            gt_x.append(inst["x"])
            gt_y.append(inst["y"])
            gt_w.append(inst["w"] - 1)
            gt_h.append(inst["h"] - 1)
            inst_starts.append(inst["start"])
            inst_ends.append(inst["end"])

        # ── STT 세그먼트 ──
        stt_segments = s["stt_segments"]
        n_stt = len(stt_segments)

        stt_embs = []
        stt_feats = []
        stt_starts, stt_ends = [], []
        for seg in stt_segments:
            stt_embs.append(self.text2emb.get(seg["text"], ZERO_EMB))
            stt_feats.append([
                seg["start"] / duration,
                seg["end"] / duration,
                (seg["end"] - seg["start"]) / duration,
            ])
            stt_starts.append(seg["start"])
            stt_ends.append(seg["end"])

        # ── 텔롭-텔롭 시간 겹침 ──
        inst_starts_arr = np.array(inst_starts)
        inst_ends_arr = np.array(inst_ends)
        overlap = (inst_starts_arr[:, None] < inst_ends_arr[None, :]) & \
                  (inst_starts_arr[None, :] < inst_ends_arr[:, None])
        overlap_mask = torch.from_numpy(overlap)  # (N, N)

        # ── 텔롭-STT 시간 겹침 ──
        if n_stt > 0:
            stt_starts_arr = np.array(stt_starts)
            stt_ends_arr = np.array(stt_ends)
            telop_stt_overlap = (inst_starts_arr[:, None] < stt_ends_arr[None, :]) & \
                                (stt_starts_arr[None, :] < inst_ends_arr[:, None])
            telop_stt_mask = torch.from_numpy(telop_stt_overlap)  # (N, S)
        else:
            telop_stt_mask = torch.zeros(n, 0, dtype=torch.bool)

        # ── 좌표 마스킹 ──
        coord_mask = torch.zeros(n, dtype=torch.bool)
        n_mask = max(1, int(n * self.mask_ratio))
        mask_idx = torch.randperm(n)[:n_mask]
        coord_mask[mask_idx] = True

        return {
            "channel_id": channel_id,
            "channel_emb": channel_emb,
            "text_embs": torch.stack(text_embs) if text_embs else torch.zeros(0, EMB_DIM),
            "cond_feats": torch.tensor(cond_feats, dtype=torch.float32),
            "gt_x": torch.tensor(gt_x, dtype=torch.long),
            "gt_y": torch.tensor(gt_y, dtype=torch.long),
            "gt_w": torch.tensor(gt_w, dtype=torch.long),
            "gt_h": torch.tensor(gt_h, dtype=torch.long),
            "coord_mask": coord_mask,
            "overlap_mask": overlap_mask,
            "stt_embs": torch.stack(stt_embs) if stt_embs else torch.zeros(0, EMB_DIM),
            "stt_feats": torch.tensor(stt_feats, dtype=torch.float32) if stt_feats else torch.zeros(0, 3),
            "telop_stt_mask": telop_stt_mask,  # (N, S)
            "n": n,
            "n_stt": n_stt,
        }


def collate_fn(batch):
    max_n = max(b["n"] for b in batch)
    max_s = max(b["n_stt"] for b in batch)
    max_s = max(max_s, 1)  # 최소 1 (전부 0이면 텐서 에러 방지)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    channel_embs = torch.zeros(B, EMB_DIM)
    text_embs = torch.zeros(B, max_n, EMB_DIM)
    cond_feats = torch.zeros(B, max_n, 4)
    gt_x = torch.zeros(B, max_n, dtype=torch.long)
    gt_y = torch.zeros(B, max_n, dtype=torch.long)
    gt_w = torch.zeros(B, max_n, dtype=torch.long)
    gt_h = torch.zeros(B, max_n, dtype=torch.long)
    coord_mask = torch.zeros(B, max_n, dtype=torch.bool)
    seq_mask = torch.zeros(B, max_n, dtype=torch.bool)
    overlap_masks = torch.zeros(B, max_n, max_n, dtype=torch.bool)

    stt_embs = torch.zeros(B, max_s, EMB_DIM)
    stt_feats = torch.zeros(B, max_s, 3)
    stt_seq_mask = torch.zeros(B, max_s, dtype=torch.bool)
    telop_stt_masks = torch.zeros(B, max_n, max_s, dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["n"]
        s = b["n_stt"]
        channel_ids[i] = b["channel_id"]
        channel_embs[i] = b["channel_emb"]
        text_embs[i, :n] = b["text_embs"]
        cond_feats[i, :n] = b["cond_feats"]
        gt_x[i, :n] = b["gt_x"]
        gt_y[i, :n] = b["gt_y"]
        gt_w[i, :n] = b["gt_w"]
        gt_h[i, :n] = b["gt_h"]
        coord_mask[i, :n] = b["coord_mask"]
        seq_mask[i, :n] = True
        overlap_masks[i, :n, :n] = b["overlap_mask"]

        if s > 0:
            stt_embs[i, :s] = b["stt_embs"]
            stt_feats[i, :s] = b["stt_feats"]
            stt_seq_mask[i, :s] = True
            telop_stt_masks[i, :n, :s] = b["telop_stt_mask"]

    return {
        "channel_id": channel_ids,
        "channel_emb": channel_embs,
        "text_embs": text_embs,
        "cond_feats": cond_feats,
        "gt_x": gt_x, "gt_y": gt_y, "gt_w": gt_w, "gt_h": gt_h,
        "coord_mask": coord_mask,
        "seq_mask": seq_mask,
        "overlap_mask": overlap_masks,
        "stt_embs": stt_embs,
        "stt_feats": stt_feats,
        "stt_seq_mask": stt_seq_mask,
        "telop_stt_mask": telop_stt_masks,
    }


train_ds = BLTDataset(train_samples, channel2id, text2emb, mask_ratio=1.0)
eval_ds = BLTDataset(eval_samples, channel2id, text2emb, mask_ratio=1.0)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# %% 셀 3: BLT 모델 정의
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
D_FF = 512
DROPOUT = 0.1

N_X = GRID_W   # 80
N_Y = GRID_H   # 80
N_W = GRID_W   # 80
N_H = GRID_H   # 80


class BLTLayoutModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.n_heads = n_heads

        # ── 텔롭 인코딩 ──
        self.channel_id_emb = nn.Embedding(n_channels, d_model)
        self.channel_text_proj = nn.Linear(emb_dim, d_model)
        self.text_proj = nn.Linear(emb_dim, d_model)

        self.time_proj = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )
        self.textlen_proj = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

        self.x_emb = nn.Embedding(N_X + 1, d_model // 4)
        self.y_emb = nn.Embedding(N_Y + 1, d_model // 4)
        self.w_emb = nn.Embedding(N_W + 1, d_model // 4)
        self.h_emb = nn.Embedding(N_H + 1, d_model // 4)

        self.MASK_X = N_X
        self.MASK_Y = N_Y
        self.MASK_W = N_W
        self.MASK_H = N_H

        self.coord_proj = nn.Linear(d_model, d_model)
        self.token_norm = nn.LayerNorm(d_model)

        # ── STT 인코딩 ──
        self.stt_emb_proj = nn.Linear(emb_dim, d_model)
        self.stt_time_proj = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )
        self.stt_norm = nn.LayerNorm(d_model)

        # ── STT cross-attention ──
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_norm = nn.LayerNorm(d_model)
        self.cross_ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.cross_ff_norm = nn.LayerNorm(d_model)

        # ── 텔롭 self-attention ──
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # ── 예측 heads ──
        self.head_x = nn.Linear(d_model, N_X)
        self.head_y = nn.Linear(d_model, N_Y)
        self.head_w = nn.Linear(d_model, N_W)
        self.head_h = nn.Linear(d_model, N_H)

        # learnable loss weights
        self.log_var_x = nn.Parameter(torch.zeros(1))
        self.log_var_y = nn.Parameter(torch.zeros(1))
        self.log_var_w = nn.Parameter(torch.zeros(1))
        self.log_var_h = nn.Parameter(torch.zeros(1))

    def encode_coords(self, gt_x, gt_y, gt_w, gt_h, coord_mask):
        x_ids = gt_x.clone()
        y_ids = gt_y.clone()
        w_ids = gt_w.clone()
        h_ids = gt_h.clone()

        x_ids[coord_mask] = self.MASK_X
        y_ids[coord_mask] = self.MASK_Y
        w_ids[coord_mask] = self.MASK_W
        h_ids[coord_mask] = self.MASK_H

        xe = self.x_emb(x_ids)
        ye = self.y_emb(y_ids)
        we = self.w_emb(w_ids)
        he = self.h_emb(h_ids)

        coord = torch.cat([xe, ye, we, he], dim=-1)
        return self.coord_proj(coord)

    def forward(self, channel_id, channel_emb, text_embs, cond_feats,
                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask,
                overlap_mask, stt_embs, stt_feats, stt_seq_mask, telop_stt_mask):
        B, N, _ = text_embs.shape
        S = stt_embs.shape[1]

        # ① 텔롭 토큰 구성
        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, N, -1)
        ch_text = self.channel_text_proj(channel_emb).unsqueeze(1).expand(-1, N, -1)
        text = self.text_proj(text_embs)
        time = self.time_proj(cond_feats[:, :, 1:])
        tlen = self.textlen_proj(cond_feats[:, :, :1])
        coord = self.encode_coords(gt_x, gt_y, gt_w, gt_h, coord_mask)

        tokens = ch_id + ch_text + text + time + tlen + coord
        tokens = self.token_norm(tokens)

        # ② STT 토큰 구성
        stt_tokens = self.stt_emb_proj(stt_embs) + self.stt_time_proj(stt_feats)
        stt_tokens = self.stt_norm(stt_tokens)

        # ③ Cross-attention: 텔롭(query) → STT(key/value)
        has_stt = stt_seq_mask.any(dim=-1)  # (B,)

        if has_stt.any():
            cross_mask = ~telop_stt_mask  # (B, N, S) True=차단

            # 겹치는 STT가 없는 텔롭 인스턴스 기록
            no_overlap = cross_mask.all(dim=-1)  # (B, N) True=겹치는 STT 없음

            # NaN 방지: 전부 차단된 행은 풀어줌
            cross_mask[no_overlap] = False

            cross_mask_expanded = cross_mask.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
            cross_mask_expanded = cross_mask_expanded.reshape(B * self.n_heads, N, S)

            stt_padding = ~stt_seq_mask  # (B, S)
            # STT 없는 샘플은 padding 해제 (NaN 방지)
            stt_padding[~has_stt] = False

            cross_out, _ = self.cross_attn(
                tokens, stt_tokens, stt_tokens,
                attn_mask=cross_mask_expanded,
                key_padding_mask=stt_padding,
            )

            # STT 없는 샘플 → 0
            cross_out = cross_out * has_stt.unsqueeze(-1).unsqueeze(-1).float()
            # 겹치는 STT 없는 텔롭 인스턴스 → 0
            cross_out = cross_out * (~no_overlap).unsqueeze(-1).float()

            tokens = self.cross_norm(tokens + cross_out)
            tokens = self.cross_ff_norm(tokens + self.cross_ff(tokens))
        else:
            tokens = self.cross_norm(tokens)
            tokens = self.cross_ff_norm(tokens + self.cross_ff(tokens))

        # ④ Self-attention (텔롭 간, 시간 겹침 mask)
        attn_mask = ~overlap_mask
        diag = torch.eye(N, dtype=torch.bool, device=attn_mask.device)
        attn_mask = attn_mask & ~diag.unsqueeze(0)
        padding_rows = ~seq_mask
        attn_mask[padding_rows] = False

        attn_mask = attn_mask.unsqueeze(1).expand(-1, self.n_heads, -1, -1)
        attn_mask = attn_mask.reshape(B * self.n_heads, N, N)

        padding_mask = ~seq_mask
        out = self.transformer(tokens, mask=attn_mask, src_key_padding_mask=padding_mask)

        # ⑤ 예측
        logits_x = self.head_x(out)
        logits_y = self.head_y(out)
        logits_w = self.head_w(out)
        logits_h = self.head_h(out)

        return logits_x, logits_y, logits_w, logits_h


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BLTLayoutModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_blt_80x80_stt"
os.makedirs(SAVE_DIR, exist_ok=True)


def gaussian_soft_label(gt, n_classes, sigma=2.0):
    device = gt.device
    arange = torch.arange(n_classes, device=device).float()
    gt_f = gt.float().unsqueeze(-1)
    label = torch.exp(-0.5 * ((arange - gt_f) / sigma) ** 2)
    label = label / label.sum(dim=-1, keepdim=True)
    return label


def compute_loss(model, logits_x, logits_y, logits_w, logits_h,
                 gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return torch.tensor(0.0, device=logits_x.device, requires_grad=True)

    soft_x = gaussian_soft_label(gt_x[valid], N_X, sigma=2.0)
    soft_y = gaussian_soft_label(gt_y[valid], N_Y, sigma=2.0)
    soft_w = gaussian_soft_label(gt_w[valid], N_W, sigma=2.0)
    soft_h = gaussian_soft_label(gt_h[valid], N_H, sigma=1.0)

    loss_x = F.cross_entropy(logits_x[valid], soft_x)
    loss_y = F.cross_entropy(logits_y[valid], soft_y)
    loss_w = F.cross_entropy(logits_w[valid], soft_w)
    loss_h = F.cross_entropy(logits_h[valid], soft_h)

    total = (loss_x / (2 * torch.exp(model.log_var_x)) + model.log_var_x / 2 +
             loss_y / (2 * torch.exp(model.log_var_y)) + model.log_var_y / 2 +
             loss_w / (2 * torch.exp(model.log_var_w)) + model.log_var_w / 2 +
             loss_h / (2 * torch.exp(model.log_var_h)) + model.log_var_h / 2)

    return total


@torch.no_grad()
def compute_metrics(logits_x, logits_y, logits_w, logits_h,
                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask):
    valid = coord_mask & seq_mask

    if valid.sum() == 0:
        return {"mae_x_px": 0, "mae_y_px": 0, "mae_w_px": 0, "mae_h_px": 0,
                "mae_x_grid": 0, "mae_y_grid": 0, "mae_w_grid": 0, "mae_h_grid": 0,
                "acc_x": 0, "acc_y": 0, "acc_w": 0, "acc_h": 0}

    pred_x = logits_x[valid].argmax(dim=-1)
    pred_y = logits_y[valid].argmax(dim=-1)
    pred_w = logits_w[valid].argmax(dim=-1)
    pred_h = logits_h[valid].argmax(dim=-1)

    mae_x_grid = (pred_x - gt_x[valid]).abs().float().mean().item()
    mae_y_grid = (pred_y - gt_y[valid]).abs().float().mean().item()
    mae_w_grid = (pred_w - gt_w[valid]).abs().float().mean().item()
    mae_h_grid = (pred_h - gt_h[valid]).abs().float().mean().item()

    return {
        "mae_x_px": mae_x_grid * CELL_SIZE_X,
        "mae_y_px": mae_y_grid * CELL_SIZE_Y,
        "mae_w_px": mae_w_grid * CELL_SIZE_X,
        "mae_h_px": mae_h_grid * CELL_SIZE_Y,
        "mae_x_grid": mae_x_grid,
        "mae_y_grid": mae_y_grid,
        "mae_w_grid": mae_w_grid,
        "mae_h_grid": mae_h_grid,
        "acc_x": (pred_x == gt_x[valid]).float().mean().item(),
        "acc_y": (pred_y == gt_y[valid]).float().mean().item(),
        "acc_w": (pred_w == gt_w[valid]).float().mean().item(),
        "acc_h": (pred_h == gt_h[valid]).float().mean().item(),
    }


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ── train ──
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        channel_emb = batch["channel_emb"].to(device)
        text_embs = batch["text_embs"].to(device)
        cond_feats = batch["cond_feats"].to(device)
        gt_x = batch["gt_x"].to(device)
        gt_y = batch["gt_y"].to(device)
        gt_w = batch["gt_w"].to(device)
        gt_h = batch["gt_h"].to(device)
        coord_mask = batch["coord_mask"].to(device)
        seq_mask = batch["seq_mask"].to(device)
        overlap_mask = batch["overlap_mask"].to(device)
        stt_embs = batch["stt_embs"].to(device)
        stt_feats = batch["stt_feats"].to(device)
        stt_seq_mask = batch["stt_seq_mask"].to(device)
        telop_stt_mask = batch["telop_stt_mask"].to(device)

        lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask,
                                overlap_mask, stt_embs, stt_feats, stt_seq_mask, telop_stt_mask)
        loss = compute_loss(model, lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    # ── eval ──
    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {k: [] for k in ["mae_x_px", "mae_y_px", "mae_w_px", "mae_h_px",
                                    "mae_x_grid", "mae_y_grid", "mae_w_grid", "mae_h_grid",
                                    "acc_x", "acc_y", "acc_w", "acc_h"]}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            channel_emb = batch["channel_emb"].to(device)
            text_embs = batch["text_embs"].to(device)
            cond_feats = batch["cond_feats"].to(device)
            gt_x = batch["gt_x"].to(device)
            gt_y = batch["gt_y"].to(device)
            gt_w = batch["gt_w"].to(device)
            gt_h = batch["gt_h"].to(device)
            coord_mask = batch["coord_mask"].to(device)
            seq_mask = batch["seq_mask"].to(device)
            overlap_mask = batch["overlap_mask"].to(device)
            stt_embs = batch["stt_embs"].to(device)
            stt_feats = batch["stt_feats"].to(device)
            stt_seq_mask = batch["stt_seq_mask"].to(device)
            telop_stt_mask = batch["telop_stt_mask"].to(device)

            lx, ly, lw, lh = model(channel_id, channel_emb, text_embs, cond_feats,
                                    gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask,
                                    overlap_mask, stt_embs, stt_feats, stt_seq_mask, telop_stt_mask)
            loss = compute_loss(model, lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)
            metrics = compute_metrics(lx, ly, lw, lh, gt_x, gt_y, gt_w, gt_h, coord_mask, seq_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    wx = 1 / (2 * torch.exp(model.log_var_x)).item()
    wy = 1 / (2 * torch.exp(model.log_var_y)).item()
    ww = 1 / (2 * torch.exp(model.log_var_w)).item()
    wh = 1 / (2 * torch.exp(model.log_var_h)).item()

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"mae: x={avg_m['mae_x_grid']:.2f} y={avg_m['mae_y_grid']:.2f} "
        f"w={avg_m['mae_w_grid']:.2f} h={avg_m['mae_h_grid']:.2f}  "
        f"px: x={avg_m['mae_x_px']:.0f} y={avg_m['mae_y_px']:.0f} "
        f"w={avg_m['mae_w_px']:.0f} h={avg_m['mae_h_px']:.0f}  "
        f"acc: x={avg_m['acc_x']:.3f} y={avg_m['acc_y']:.3f} "
        f"w={avg_m['acc_w']:.3f} h={avg_m['acc_h']:.3f}  "
        f"wt: x={wx:.2f} y={wy:.2f} w={ww:.2f} h={wh:.2f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")